# Внимание, позиции и маски

Авторы BelOAI. CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи, затем обратитесь к решениям.

## Чему научимся

Вычислять веса внимания и проверять причинную маску без доступа к будущим токенам.

## К каким позициям обратиться

Внимание строит новый вектор позиции как взвешенную сумму значений других позиций. Query Q — запрос, key K — описание для сравнения, value V — передаваемое содержимое. Эти английские названия означают роли в вычислении, а не отдельные базы данных.

В scaled dot-product attention счёты равны $QK^T/\sqrt d$, где d — длина запроса и ключа. Softmax по доступным ключам даёт веса, затем их умножают на V. Self-attention получает Q,K,V из одной последовательности с помощью обучаемых преобразований.

Например, запрос $q=(1,0)$ и два ключа $k_1=(1,0)$, $k_2=(0,1)$ дают скалярные произведения 1 и 0. При d=2 счёты равны $[1/\sqrt2,0]$. После softmax веса приблизительно [0.67,0.33]: первый ключ получил большее скалярное произведение с запросом. Это не утверждение о меньшем евклидовом расстоянии: внимание использует именно указанные счёты. Значения V могут отличаться от ключей: по ключам выбираем, а усредняем содержимое значений. Одну такую операцию называют головой внимания.

## Разобранный пример: два значения

Один запрос даёт счёты [0,ln3]. Softmax равен [1/4,3/4]. При значениях [2,10] ответ 0.25·2+0.75·10=8. Это смешивание доступного содержимого; по одному весу 0.75 нельзя уверенно объяснить, почему вся модель приняла решение.

Если модель предсказывает следующий токен, позиции нельзя видеть будущие ответы. **Причинная маска** запрещает обращения вправо: недопустимым счётам до softmax ставят минус бесконечность, их веса становятся нулевыми. Для полностью запрещённой строки softmax не даёт корректного распределения, поэтому нужно оставлять допустимые позиции и отдельно учитывать заполнение коротких последовательностей (padding).

Маска не удаляет позиции из последовательности, а запрещает конкретные обращения. Для трёх позиций первая видит только первую, вторая — первую и вторую, третья — все три. Своя позиция разрешена, потому что при обучении из уже известного токена предсказывают **следующий**, а не пытаются восстановить его самого по подсказке с ответом.

## Порядок и стоимость

Само сравнение токенов без позиционных сведений не кодирует обычный порядок текста. Позиционные представления добавляют информацию о месте. При длине L таблица всех пар внимания имеет L² элементов на голову; длинный контекст требует памяти даже при параллельном вычислении.

В ноутбуке torch.triu создаёт верхний треугольник маски, masked_fill ставит запрещённым элементам −∞, softmax нормирует строки. Это одна маленькая голова внимания без обученной языковой модели.

## Разобранный пример: проверяем вычисление

Все числа в примере придуманы для этого урока. Выполните код, затем измените вход и объясните новый результат.

Код считает сразу три запроса: Q и K имеют форму (3,2), V — (3,1). `diagonal=1` у `triu` оставляет запрещённой часть строго выше главной диагонали. Поэтому первый запрос видит только своё значение 2, а его веса обязаны быть [1,0,0]. Это другой числовой пример, чем заранее заданные веса [0.25,0.75] выше.

In [ ]:
import torch
import math
torch.set_num_threads(1)
Q = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])
K = Q.clone()
V = torch.tensor([[2.], [4.], [8.]])
scores = Q @ K.T / math.sqrt(Q.shape[1])
forbidden = torch.triu(torch.ones(3, 3, dtype=torch.bool), diagonal=1)
weights = torch.softmax(scores.masked_fill(forbidden, float("-inf")), dim=1)
output = weights @ V
assert torch.allclose(weights.sum(dim=1), torch.ones(3))
assert torch.all(weights[forbidden] == 0)
assert output.shape == (3, 1)
assert torch.allclose(weights[0], torch.tensor([1., 0., 0.]))
assert output[0, 0].item() == 2.
print(weights, output)


## Типичные ошибки

Не пропускайте деление на √d и ось softmax. Не разрешайте будущее и не оставляйте полностью закрытую строку.

## Задачи

### 1. Базовая

По какой оси нормируют веса одного запроса?

<details>
<summary>Подсказка</summary>

Сравниваются доступные ключи.

</details>



### 2. Базовая

Зачем причинная маска?

<details>
<summary>Подсказка</summary>

Нельзя узнать ответ из будущего.

</details>



### 3. Прикладная

Веса [0.25,0.75], значения [2,10]. Найдите ответ.

<details>
<summary>Подсказка</summary>

Взвешенная сумма.

</details>



### 4. Прикладная

Длина выросла со 100 до 200. Как растёт таблица пар?

<details>
<summary>Подсказка</summary>

Возведите коэффициент в квадрат.

</details>



### 5. Олимпиадная
Softmax дал веса [0.25,0.75] для значений [2,10]. Вторую позицию нужно запретить. Сравните ответ при простом обнулении её веса и при повторной нормировке. Почему удобнее наложить маску до softmax?

<details>
<summary>Подсказка</summary>

После удаления позиции сумма оставшихся весов должна снова быть равна 1.

</details>



### 6. Олимпиадная

Доказывает ли большой вес внимания причинное объяснение ответа?

<details>
<summary>Подсказка</summary>

Ответ зависит от многих слоёв и значений.

</details>



## Итоги и самопроверка

Проследите Q,K,V до результата и укажите форму каждой таблицы. Что запрещает маска?

Для перехода дальше достаточно базовых и прикладных задач. Олимпиадные задачи можно отложить.


## Решения задач

### Задача 1

По позициям ключей, чтобы их веса суммировались в 1.

### Задача 2

Чтобы позиция не использовала последующие токены при авторегрессионном прогнозе.

### Задача 3

0.5+7.5=8.

### Задача 4

В четыре раза.

### Задача 5

При простом обнулении получаются веса [0.25,0], ответ 0.25·2=0.5. После нормировки оставшегося веса получаем [1,0], ответ 2. Разрешена только первая позиция, поэтому именно её значение должно стать взвешенным средним. Маска до softmax сразу нормирует по доступным позициям. Если запрещены все позиции, такое среднее не определено: этот случай нужно отклонить или обработать отдельно.

### Задача 6

Нет. Вес — промежуточная величина, которую нельзя автоматически трактовать как полное объяснение.